# CS3807 Deep Learning Laboratory - Experiment 4

**Comparative Study of Deep Convolutional Neural Network Architectures Using Transfer Learning**

This notebook implements the CIFAR-10 transfer-learning experiment, fine-tuning, the complete hyperparameter study, all mandatory plots, evaluation metrics, VGG16 and ResNet50 transfer-learning exercises, Adam vs SGD, frozen vs partial fine-tuning, and a controlled CIFAR-10 comparison of LeNet-5, AlexNet, GoogleNet, VGG16 and ResNet50.

In [ ]:
import os, time, json, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)
CLASS_NAMES = ['Airplane','Automobile','Bird','Cat','Deer','Dog','Frog','Horse','Ship','Truck']
print('TensorFlow:', tf.__version__)


In [ ]:
(x_all, y_all), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_all = y_all.ravel(); y_test = y_test.ravel()
x_train, x_val, y_train, y_val = train_test_split(
    x_all, y_all, test_size=5000, random_state=SEED, stratify=y_all)
print('Training:', x_train.shape, 'Validation:', x_val.shape, 'Testing:', x_test.shape)
print('Pixel range before preprocessing:', int(x_train.min()), int(x_train.max()))


In [ ]:
fig, axes = plt.subplots(2,5,figsize=(10,4.4))
for cls, ax in enumerate(axes.ravel()):
    idx = np.flatnonzero(y_all == cls)[0]
    ax.imshow(x_all[idx]); ax.set_title(CLASS_NAMES[cls], fontsize=9); ax.axis('off')
fig.suptitle('CIFAR-10: One Sample from Each Class', fontsize=12)
plt.tight_layout(rect=[0,0,1,0.94])
plt.savefig(OUT/'sample_cifar10_images.png', dpi=220, bbox_inches='tight'); plt.show()


## Main Transfer-Learning Experiment - MobileNetV2

In [ ]:
mobile_preprocess = keras.applications.mobilenet_v2.preprocess_input
mobile_base = keras.applications.MobileNetV2(weights='imagenet', include_top=False,
                                             input_shape=(32,32,3), pooling='avg')
mobile_base.trainable = False

def extract_features(base, preprocess, x, name, batch_size=256):
    t0=time.perf_counter(); f=base.predict(preprocess(x.astype('float32')), batch_size=batch_size, verbose=0)
    elapsed=time.perf_counter()-t0
    print(name, f.shape, f'{elapsed:.2f} s')
    return f, elapsed

train_features,tft = extract_features(mobile_base,mobile_preprocess,x_train,'Train')
val_features,tfv = extract_features(mobile_base,mobile_preprocess,x_val,'Validation')
test_features,tfs = extract_features(mobile_base,mobile_preprocess,x_test,'Test')
feature_extraction_time=tft+tfv+tfs


In [ ]:
def make_head(units=128, optimizer='Adam', learning_rate=0.001):
    m=keras.Sequential([layers.Input((train_features.shape[1],)),
                        layers.Dense(units,activation='relu'), layers.Dropout(0.2),
                        layers.Dense(10,activation='softmax')])
    opt = keras.optimizers.Adam(learning_rate) if optimizer=='Adam' else keras.optimizers.SGD(learning_rate, momentum=0.9)
    m.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

head=make_head(128,'Adam',0.001)
t0=time.perf_counter()
history_head=head.fit(train_features,y_train,validation_data=(val_features,y_val),epochs=10,batch_size=32,verbose=2)
head_training_time=time.perf_counter()-t0
before_loss,before_acc=head.evaluate(test_features,y_test,batch_size=256,verbose=0)
print('Frozen-base test accuracy:',before_acc)


## Hyperparameter Study

In [ ]:
hyper_rows=[]
def run_head_setting(label,lr=0.001,batch=32,epochs=10,optimizer='Adam',units=128):
    m=make_head(units,optimizer,lr); t0=time.perf_counter()
    h=m.fit(train_features,y_train,validation_data=(val_features,y_val),epochs=epochs,batch_size=batch,verbose=0)
    elapsed=time.perf_counter()-t0
    loss,acc=m.evaluate(test_features,y_test,batch_size=256,verbose=0)
    row={'Setting':label,'Learning Rate':lr,'Batch Size':batch,'Epochs':epochs,'Optimizer':optimizer,
         'Dense Units':units,'Frozen Layers':'All','Best Validation Accuracy':float(max(h.history['val_accuracy'])),
         'Test Accuracy':float(acc),'Training Time (s)':float(elapsed)}
    hyper_rows.append(row); del m; gc.collect(); return h,row

_,hp_base=run_head_setting('Baseline',0.001,32,10,'Adam',128)
_,hp_lr=run_head_setting('Learning rate 0.0001',0.0001,32,10,'Adam',128)
_,hp_b16=run_head_setting('Batch size 16',0.001,16,10,'Adam',128)
_,hp_b64=run_head_setting('Batch size 64',0.001,64,10,'Adam',128)
h20,hp_e20=run_head_setting('Epochs 20',0.001,32,20,'Adam',128)
_,hp_sgd=run_head_setting('Optimizer SGD',0.001,32,10,'SGD',128)
_,hp_u256=run_head_setting('Dense units 256',0.001,32,10,'Adam',256)
hyper_df=pd.DataFrame(hyper_rows); hyper_df.to_csv(OUT/'hyperparameter_study.csv',index=False)
hyper_df


## Fine-Tuning

In [ ]:
inputs=keras.Input((32,32,3)); x=mobile_preprocess(inputs); x=mobile_base(x,training=False)
x=layers.Dense(128,activation='relu',name='classifier_dense')(x); x=layers.Dropout(0.2)(x)
outputs=layers.Dense(10,activation='softmax',name='classifier_output')(x)
mobile_model=keras.Model(inputs,outputs,name='MobileNetV2_CIFAR10')
mobile_model.get_layer('classifier_dense').set_weights(head.layers[0].get_weights())
mobile_model.get_layer('classifier_output').set_weights(head.layers[2].get_weights())
mobile_base.trainable=True
for layer in mobile_base.layers[:-20]: layer.trainable=False
for layer in mobile_base.layers[-20:]:
    if isinstance(layer,layers.BatchNormalization): layer.trainable=False
mobile_model.compile(optimizer=keras.optimizers.Adam(1e-5),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
t0=time.perf_counter()
history_ft=mobile_model.fit(x_train,y_train,validation_data=(x_val,y_val),epochs=5,batch_size=32,verbose=2)
fine_tune_time=time.perf_counter()-t0
final_loss,final_acc=mobile_model.evaluate(x_test,y_test,batch_size=256,verbose=0)
print('Fine-tuned test accuracy:',final_acc)
hyper_rows.append({'Setting':'Frozen layers partial','Learning Rate':1e-5,'Batch Size':32,'Epochs':5,
                   'Optimizer':'Adam','Dense Units':128,'Frozen Layers':'Partial',
                   'Best Validation Accuracy':float(max(history_ft.history['val_accuracy'])),
                   'Test Accuracy':float(final_acc),'Training Time (s)':float(fine_tune_time)})
hyper_df=pd.DataFrame(hyper_rows); hyper_df.to_csv(OUT/'hyperparameter_study.csv',index=False)
hyper_df


## Mandatory Plots

In [ ]:
epochs=np.arange(1,16)
train_acc=history_head.history['accuracy']+history_ft.history['accuracy']
val_acc=history_head.history['val_accuracy']+history_ft.history['val_accuracy']
train_loss=history_head.history['loss']+history_ft.history['loss']
val_loss=history_head.history['val_loss']+history_ft.history['val_loss']

def save_curve(values,ylabel,title,filename):
    plt.figure(figsize=(7.5,4.2)); plt.plot(epochs,values,marker='o',markersize=3)
    plt.axvline(10.5,linestyle='--',linewidth=1,label='Fine-tuning begins')
    plt.xlabel('Epoch'); plt.ylabel(ylabel); plt.title(title); plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
    plt.savefig(OUT/filename,dpi=220,bbox_inches='tight'); plt.show()
save_curve(train_acc,'Accuracy','Training Accuracy vs Epoch','training_accuracy.png')
save_curve(val_acc,'Accuracy','Validation Accuracy vs Epoch','validation_accuracy.png')
save_curve(train_loss,'Categorical Cross-Entropy Loss','Training Loss vs Epoch','training_loss.png')
save_curve(val_loss,'Categorical Cross-Entropy Loss','Validation Loss vs Epoch','validation_loss.png')


In [ ]:
t0=time.perf_counter(); y_prob=mobile_model.predict(x_test,batch_size=256,verbose=0); prediction_time=time.perf_counter()-t0
y_pred=np.argmax(y_prob,axis=1)
precision,recall,f1,_=precision_recall_fscore_support(y_test,y_pred,average='weighted',zero_division=0)
cm=confusion_matrix(y_test,y_pred)
report_df=pd.DataFrame(classification_report(y_test,y_pred,target_names=CLASS_NAMES,output_dict=True,zero_division=0)).transpose()
report_df.to_csv(OUT/'classification_report.csv')
fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(cm,cmap='Blues'); fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04)
ax.set_xticks(range(10),labels=CLASS_NAMES,rotation=45,ha='right'); ax.set_yticks(range(10),labels=CLASS_NAMES)
ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label'); ax.set_title('MobileNetV2 CIFAR-10 Confusion Matrix')
th=cm.max()/2
for i in range(10):
    for j in range(10): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=7,color='white' if cm[i,j]>th else 'black')
plt.tight_layout(); plt.savefig(OUT/'confusion_matrix.png',dpi=220,bbox_inches='tight'); plt.show()


## Controlled CNN Architecture Comparison

In [ ]:
bench_x_train,_,bench_y_train,_=train_test_split(x_train,y_train,train_size=10000,random_state=SEED,stratify=y_train)
bench_x_val,_,bench_y_val,_=train_test_split(x_val,y_val,train_size=2000,random_state=SEED,stratify=y_val)
bench_x_test,_,bench_y_test,_=train_test_split(x_test,y_test,train_size=2000,random_state=SEED,stratify=y_test)
xbt=bench_x_train.astype('float32')/255.; xbv=bench_x_val.astype('float32')/255.; xbte=bench_x_test.astype('float32')/255.
print(xbt.shape,xbv.shape,xbte.shape)


In [ ]:
def build_lenet():
    inp=keras.Input((32,32,3)); x=layers.Conv2D(6,5,activation='tanh')(inp); x=layers.AveragePooling2D(2)(x)
    x=layers.Conv2D(16,5,activation='tanh')(x); x=layers.AveragePooling2D(2)(x); x=layers.Flatten()(x)
    x=layers.Dense(120,activation='tanh')(x); x=layers.Dense(84,activation='tanh')(x); out=layers.Dense(10,activation='softmax')(x)
    return keras.Model(inp,out,name='LeNet5')

def build_alexnet():
    inp=keras.Input((32,32,3)); x=layers.Conv2D(64,3,padding='same',activation='relu')(inp); x=layers.MaxPooling2D(2)(x)
    x=layers.Conv2D(192,3,padding='same',activation='relu')(x); x=layers.MaxPooling2D(2)(x)
    x=layers.Conv2D(384,3,padding='same',activation='relu')(x); x=layers.Conv2D(256,3,padding='same',activation='relu')(x)
    x=layers.Conv2D(256,3,padding='same',activation='relu')(x); x=layers.MaxPooling2D(2)(x); x=layers.Flatten()(x)
    x=layers.Dense(1024,activation='relu')(x); x=layers.Dropout(0.5)(x); x=layers.Dense(1024,activation='relu')(x); x=layers.Dropout(0.5)(x)
    out=layers.Dense(10,activation='softmax')(x); return keras.Model(inp,out,name='AlexNet_CIFAR')

def inc(x,f1,f3r,f3,f5r,f5,proj):
    a=layers.Conv2D(f1,1,padding='same',activation='relu')(x)
    b=layers.Conv2D(f3r,1,padding='same',activation='relu')(x); b=layers.Conv2D(f3,3,padding='same',activation='relu')(b)
    c=layers.Conv2D(f5r,1,padding='same',activation='relu')(x); c=layers.Conv2D(f5,5,padding='same',activation='relu')(c)
    d=layers.MaxPooling2D(3,strides=1,padding='same')(x); d=layers.Conv2D(proj,1,padding='same',activation='relu')(d)
    return layers.Concatenate()([a,b,c,d])

def build_googlenet():
    inp=keras.Input((32,32,3)); x=layers.Conv2D(64,3,padding='same',activation='relu')(inp); x=layers.MaxPooling2D(2)(x)
    x=inc(x,64,96,128,16,32,32); x=inc(x,128,128,192,32,96,64); x=layers.MaxPooling2D(2)(x)
    x=inc(x,192,96,208,16,48,64); x=inc(x,160,112,224,24,64,64); x=inc(x,128,128,256,24,64,64)
    x=layers.GlobalAveragePooling2D()(x); x=layers.Dropout(0.4)(x); out=layers.Dense(10,activation='softmax')(x)
    return keras.Model(inp,out,name='GoogleNet_CIFAR')

def train_scratch(name,builder,epochs=3):
    tf.keras.backend.clear_session(); m=builder(); m.compile(optimizer=keras.optimizers.Adam(0.001),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    t0=time.perf_counter(); m.fit(xbt,bench_y_train,validation_data=(xbv,bench_y_val),epochs=epochs,batch_size=64,verbose=2)
    elapsed=time.perf_counter()-t0; loss,acc=m.evaluate(xbte,bench_y_test,batch_size=256,verbose=0)
    row={'Model':name,'Parameters':int(m.count_params()),'Accuracy':float(acc),'Training Time (s)':float(elapsed)}
    del m; gc.collect(); return row

arch_rows=[train_scratch('LeNet-5',build_lenet),train_scratch('AlexNet',build_alexnet),train_scratch('GoogleNet',build_googlenet)]


## VGG16 and ResNet50 Transfer-Learning Exercises

In [ ]:
def transfer_benchmark(app_builder,preprocess,model_name,unfreeze_last,epochs_frozen=3,epochs_ft=1):
    tf.keras.backend.clear_session(); base=app_builder(weights='imagenet',include_top=False,input_shape=(32,32,3),pooling='avg'); base.trainable=False
    inp=keras.Input((32,32,3)); x=preprocess(inp); x=base(x,training=False); x=layers.Dense(128,activation='relu')(x); out=layers.Dense(10,activation='softmax')(x)
    m=keras.Model(inp,out); m.compile(optimizer=keras.optimizers.Adam(0.001),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    t0=time.perf_counter(); m.fit(bench_x_train,bench_y_train,validation_data=(bench_x_val,bench_y_val),epochs=epochs_frozen,batch_size=64,verbose=2)
    frozen_loss,frozen_acc=m.evaluate(bench_x_test,bench_y_test,batch_size=256,verbose=0)
    base.trainable=True
    for layer in base.layers[:-unfreeze_last]: layer.trainable=False
    for layer in base.layers[-unfreeze_last:]:
        if isinstance(layer,layers.BatchNormalization): layer.trainable=False
    m.compile(optimizer=keras.optimizers.Adam(1e-5),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    m.fit(bench_x_train,bench_y_train,validation_data=(bench_x_val,bench_y_val),epochs=epochs_ft,batch_size=64,verbose=2)
    elapsed=time.perf_counter()-t0; loss,acc=m.evaluate(bench_x_test,bench_y_test,batch_size=256,verbose=0)
    row={'Model':model_name,'Parameters':int(m.count_params()),'Accuracy':float(acc),'Training Time (s)':float(elapsed),
         'Frozen Accuracy':float(frozen_acc),'Fine-Tuned Accuracy':float(acc)}
    del m,base; gc.collect(); return row

vgg_row=transfer_benchmark(keras.applications.VGG16,keras.applications.vgg16.preprocess_input,'VGG16',4)
resnet_row=transfer_benchmark(keras.applications.ResNet50,keras.applications.resnet50.preprocess_input,'ResNet50',10)
arch_rows.extend([vgg_row,resnet_row])
arch_df=pd.DataFrame(arch_rows); arch_df['Accuracy (%)']=arch_df['Accuracy']*100
arch_df.to_csv(OUT/'architecture_comparison.csv',index=False); arch_df


## Additional Exercise Comparisons

In [ ]:
optimizer_compare=hyper_df[hyper_df['Setting'].isin(['Baseline','Optimizer SGD'])][['Setting','Optimizer','Best Validation Accuracy','Test Accuracy','Training Time (s)']].copy()
optimizer_compare.to_csv(OUT/'adam_vs_sgd.csv',index=False)
frozen_compare=pd.DataFrame([
    {'Training Strategy':'Frozen convolutional base','Test Accuracy':float(before_acc),'Training Time (s)':float(head_training_time)},
    {'Training Strategy':'Partial fine-tuning','Test Accuracy':float(final_acc),'Training Time (s)':float(fine_tune_time)}])
frozen_compare.to_csv(OUT/'frozen_vs_finetuned.csv',index=False)
optimizer_compare, frozen_compare


In [ ]:
mobile_metrics={'Model':'MobileNetV2','Pretraining':'ImageNet','Training Images':int(len(x_train)),'Validation Images':int(len(x_val)),
'Testing Images':int(len(x_test)),'Training Accuracy':float(train_acc[-1]),'Frozen Test Accuracy':float(before_acc),'Fine-Tuned Test Accuracy':float(final_acc),
'Precision':float(precision),'Recall':float(recall),'F1-score':float(f1),'Test Loss':float(final_loss),'Feature Extraction Time (s)':float(feature_extraction_time),
'Classifier Training Time (s)':float(head_training_time),'Fine-Tuning Time (s)':float(fine_tune_time),'Total Training Time (s)':float(head_training_time+fine_tune_time),
'Prediction Time (s)':float(prediction_time),'Total Parameters':int(mobile_model.count_params()),'Confusion Matrix':cm.tolist(),
'Training Accuracy History':[float(v) for v in train_acc],'Validation Accuracy History':[float(v) for v in val_acc],
'Training Loss History':[float(v) for v in train_loss],'Validation Loss History':[float(v) for v in val_loss]}
summary={'mobile_metrics':mobile_metrics,'hyperparameter_study':hyper_df.to_dict(orient='records'),'architecture_comparison':arch_df.to_dict(orient='records'),
         'optimizer_comparison':optimizer_compare.to_dict(orient='records'),'frozen_vs_finetuned':frozen_compare.to_dict(orient='records'),
         'classification_report':report_df.reset_index().rename(columns={'index':'Class'}).to_dict(orient='records')}
with open(OUT/'all_results.json','w') as f: json.dump(summary,f,indent=2)
print(json.dumps({k:v for k,v in mobile_metrics.items() if not isinstance(v,list)},indent=2))
